# 2 - Record a dataset and stream it back

Capture a robot demonstration as a [LeRobotDataset](https://github.com/huggingface/lerobot), then read it back frame by frame with `stream_dataset()` - the in-process counterpart to recording. Camera video decodes on the fly; nothing is re-materialized to disk.

No hardware, no GPU, no Hugging Face credentials. Part of the [examples/notebooks](./README.md) series; see [notebook 1](./01_getting_started.ipynb) first.

In [ ]:
import os
# macOS uses the 'cgl' offscreen GL backend; Linux headless uses 'egl'.
os.environ.setdefault("MUJOCO_GL", "cgl")

ROOT, REPO = "/tmp/nb2_dataset", "local/nb2_demo"

## Record

`start_recording()` opens a `LeRobotDataset`, the policy loop fills it frame by frame, and `stop_recording()` finalizes it (encoding per-camera MP4 and writing the `meta/` folder of stats and schema). A camera is added first so the recorded dataset has an image stream.

In [ ]:
from strands_robots import Robot, MockPolicy

sim = Robot("so100", mesh=False)
sim.add_camera(name="front", position=[0.5, 0.0, 0.4], target=[0.2, 0.0, 0.05])

sim.start_recording(repo_id=REPO, root=ROOT, fps=30,
                    task="pick up the red cube", overwrite=True)
sim.run_policy(robot_name="so100", policy_object=MockPolicy(),
               instruction="pick up the red cube", n_steps=60)
sim.stop_recording()
print("recorded ->", ROOT)

## Stream it back

`stream_dataset()` reads the dataset you just wrote. Iterating yields one frame at a time: the joint state and action come from the Parquet shards, and camera frames decode from the MP4 shards as you go. Only the small `meta/` folder is read up front.

In [ ]:
reader = sim.stream_dataset(REPO, root=ROOT, shuffle=False)
print("episodes:", reader.num_episodes, "| frames:", reader.num_frames, "| fps:", reader.fps)

for n, frame in enumerate(reader):
    if n == 0:
        cams = [k for k in frame if k.startswith("observation.images.")]
        print("state:", tuple(frame["observation.state"].shape),
              "| action:", tuple(frame["action"].shape),
              "| cameras:", cams)
    if n >= 4:
        break
print("streamed frames OK - nothing re-downloaded")
sim.destroy()

## Why this matters

The same `Robot()` that wrote the dataset reads it back, so collecting data and consuming it are two methods on one object in one format. For training at scale, hand the reader a `DataLoader` (`reader.dataloader(batch_size=64)`); for storing collection runs, `stop_recording(bucket="org/name")` syncs to a Hugging Face Storage Bucket.

## Next

**[3 - Record, train, deploy](./03_record_train_deploy.ipynb)** trains a policy on a dataset like this one.